# Reading a CSV

In [ ]:
import Data.Csv (FromRecord(..), ToRecord, HasHeader(..))
import GHC.Generics
import System.IO
import System.Exit (exitFailure)
-- from bytestring
import Data.ByteString (ByteString, hGetSome, empty)
import qualified Data.ByteString.Lazy as BL
import Data.Csv.Incremental (Parser(..), decode)

In [ ]:
import Data.Either (lefts, rights)

In [ ]:
import Numeric.LinearAlgebra ((<\>), takeDiag, inv, tr, dot, (<#), ident, (!), svd)
import qualified Numeric.LinearAlgebra as LA ((<>))
import Numeric.LinearAlgebra.Data (fromLists, fromList, Vector, rows)

In [ ]:
data SpectorRecord = SpectorRecord
  { obs :: !Int
  , gpa  :: !Double
  , tuce :: !Int
  , psi :: !Int
  , grade :: !Int
  } deriving (Show, Eq, Generic)

instance FromRecord SpectorRecord
instance ToRecord SpectorRecord


In [ ]:
:info parseRecord

In [ ]:
feed :: (ByteString -> Parser SpectorRecord) -> Handle -> IO (Parser SpectorRecord)
feed k csvFile = do
  hIsEOF csvFile >>= \eof -> case eof of
    True  -> return $ k empty
    False -> k <$> hGetSome csvFile 4096

spector <- withFile "spector.csv" ReadMode $ \ csvFile -> do
  let loop !_ (Fail _ errMsg) = do putStrLn errMsg; return []
      loop acc (Many rs k)    = loop (acc <> rs) =<< feed k csvFile
      loop acc (Done rs)      = return $ (acc <> rs)
  loop [] (decode HasHeader)


In [ ]:
rights spector

In [ ]:
x0 = fromLists $ map (\spector -> [1.0, gpa spector, fromIntegral $ tuce spector, fromIntegral $ psi spector]) (rights spector)
y = fromList $ map (fromIntegral . grade) (rights spector) :: Vector Double

In [ ]:
import Numeric.LinearAlgebra (thinSVD, disp, diag, diagRect)
import Numeric.LinearAlgebra.Data (rows, cols)

In [ ]:
import Numeric.LinearAlgebra.Data (maxElement, cmap, subVector, conj)
import Numeric.LinearAlgebra (Matrix)

In [ ]:
-- import Numeric.LinearAlgebra ((><))
-- let a = (5><3) [  1.0,  2.0,  3.0
--      ,  4.0,  5.0,  6.0
--      ,  7.0,  8.0,  9.0
--      , 10.0, 11.0, 12.0
--      , 13.0, 14.0, 15.0 ] :: Matrix Double
--     (u,s,v) = svd a
-- disp 3 $ u <> (diagRect 0 s 5 3) <> tr v

In [ ]:
-- pinv_extended :: Matrix Double -> Double -> ?
-- svd :: Field t => Matrix t -> (Matrix t, Vector Double, Matrix t)
let (u, s, v) = svd $ conj x0
disp 2 u
disp 2 $ diagRect 0 s 4 32
disp 2 v

In [ ]:
tr x0

In [ ]:
disp 4 $ u <> diagRect 0 s 32 4 <> tr v

In [ ]:
disp 4 $ conj x0 - (u <> diagRect 0 s 32 4 <> tr v)

In [ ]:
disp 2 $ v <> tr v

In [ ]:
-- pinv_extended :: Matrix Double -> Double -> ?
let (u, s, v) = thinSVD $ conj x0
disp 4 u
disp 4 $ diagRect 0 s 4 4
disp 4 v

In [ ]:
pinvExtended :: Matrix Double -> Double -> (Matrix Double, Vector Double)
pinvExtended m rcond = (v <> diagRect 0 srecip rrow rcol <> tr u, s)
    where (u, s, v) = thinSVD $ conj x0
          rcol = cols u
          rrow = cols v
          urow = rows u
          vrow = rows v
          cutoff = rcond * maxElement s
          srecip = cmap (\x -> if x > cutoff then 1.0 / x else 0.0) (subVector 0 (min urow vrow) s)

-- print cutoff
-- let srecip = cmap (\x -> if x > cutoff then 1.0 / x else 0.0) (subVector 0 (min (rows u) (rows v)) s)
-- print srecip
-- let ret = v <> diagRect 0 srecip (cols v) (rows u) <> tr u
-- disp 4 u
-- disp 4 $ diagRect 0 s 4 32
-- disp 4 v

In [ ]:
let (xinv, s) = pinvExtended x0 1e-15
disp 3 $ xinv <> x0 -- identity
disp 3 $ x0 <> xinv 

In [ ]:
-- import Numeric.LinearAlgebra.Data (maxElement, cmap, subVector)
import Numeric.LinearAlgebra.HMatrix (rank, app)
import Numeric.LinearAlgebra (norm_2)

In [ ]:
-- TODO: need to figure out the pre-processing to get wexog
let wexog = x0
    wendog = y -- TODO: I think
    (pinvWexog, singularValues) = pinvExtended wexog 1e-15
    normalized_cov_params = pinvWexog <> tr pinvWexog -- TODO: Should this be tr'?
    -- Cache these singular values for use later.
    wexogSingularValues = singularValues
    dataRank = rank $ diag singularValues -- following statsmodels, but is this just the number of non-zero singular values, possibly with a cutoff
    beta = pinvWexog `app` wendog


In [ ]:
class Model m x y where
    predict :: m -> x -> y
    
class ModelEstimator f x y r where
    fit :: f -> x -> y -> r

In [ ]:
data CovarianceType = NonRobustCovariance
  deriving (Eq, Show)

data OLSResults = OLSResults 
    { olsResults_beta :: Vector Double
    , olsResults_normalizedCovParams :: Matrix Double
    , olsResults_covarianceType :: CovarianceType
    , olsResults_uset :: Bool
    , olsResults_trainingData :: (Matrix Double, Vector Double) } deriving (Eq, Show)


In [ ]:
instance Model OLSResults (Matrix Double) (Vector Double) where
    predict olsres x = x `app` olsResults_beta olsres

In [ ]:
:info y

In [ ]:
fitOLS :: Matrix Double -> Vector Double -> OLSResults
fitOLS x y = let 
    wexog = x
    wendog = y
    (pinvWexog, singularValues) = pinvExtended wexog 1e-15
    normalized_cov_params = pinvWexog <> tr pinvWexog -- TODO: Should this be tr'?
    -- Cache these singular values for use later.
    wexogSingularValues = singularValues
    dataRank = rank $ diag singularValues -- following statsmodels, but is this just the number of non-zero singular values, possibly with a cutoff
    beta = pinvWexog `app` wendog
    in OLSResults beta normalized_cov_params NonRobustCovariance True (wexog, wendog)

In [ ]:
let ols = fitOLS x0 y
ols

In [ ]:
:type (predict ols x0 :: Vector Double)

In [ ]:
predict ols x0 :: Vector Double

In [ ]:
-- import Numeric.LinearAlgebra.Static (mean)
import Numeric.LinearAlgebra.Data (size, konst, scalar)
import Numeric.LinearAlgebra (sumElements, Container)
import Data.Vector (length)

In [ ]:
:type konst

In [ ]:
konst 7 3 :: Vector Double

In [ ]:
fromList [1..5] - scalar 5.0

In [ ]:
(fromIntegral . size) $ fromList [1..5]
sumElements (fromList [1..5]) / (fromIntegral . size) (fromList [1..5])

In [ ]:
-- Had to import Container for this.  Not sure if importing length would work too.
mean :: (Container Vector e, Fractional e) => Vector e -> e
mean v | size v > 0 = sumElements v / (fromIntegral . size) v
       | otherwise = 0
-- mean v | vsize > 0 = sumElements v / (fromIntegral . size) v
--        | otherwise = 0.0
--     where vsize = size v :: Int

In [ ]:
nobs :: OLSResults -> Int
nobs ols = let
    (wexog, _) = olsResults_trainingData ols
    in rows wexog

    
fittedvalues :: OLSResults -> Vector Double
fittedvalues ols = let
    (wexog, _) = olsResults_trainingData ols
    in predict ols wexog

-- NOTE: For OLS, we should consider dropping wexog, wendog, wresid, etc.
wresid :: OLSResults -> Vector Double
wresid ols = let
    (wexog, wendog) = olsResults_trainingData ols
    in wendog - predict ols wexog

ssr :: OLSResults -> Double
ssr = norm_2 . wresid

centeredTSS :: OLSResults -> Double
centeredTSS ols = let
    (_, wendog) = olsResults_trainingData ols
    r = size wendog
    m = mean wendog
    in norm_2 $ wendog - konst m r

uncenteredTSS :: OLSResults -> Double
uncenteredTSS ols = norm_2 wendog
    where (_, wendog) = olsResults_trainingData ols

-- TODO: The boolean is 
rsquared :: OLSResults -> Bool -> Double
rsquared ols hasconst = if hasconst
    then 1.0 - ssr ols / centeredTSS ols
    else 1.0 - ssr ols / uncenteredTSS ols

-- The explained sum of squares.
-- If a constant is present, the centered total sum of squares minus the
-- sum of squared residuals. If there is no constant, the uncentered total
-- sum of squares is used.
ess :: OLSResults -> Bool -> Double
ess ols hasconst = if hasconst
    then centeredTSS ols - ssr ols
    else uncenteredTSS ols - ssr ols


In [ ]:
-- TODO: Some additional work is needed here.
-- Adjusted R-squared.
-- This is defined here as 1 - (`nobs`-1)/`df_resid` * (1-`rsquared`)
-- if a constant is included and 1 - `nobs`/`df_resid` * (1-`rsquared`) if
-- no constant is included.
-- rsquared_adj :: OLSResults -> Bool -> Double
-- rsquared_adj ols hasconst = 1 - adj * (1 - rsquared ols hasconst)
--     where k_constant = if hasconst then 1 else 0
--           adj = (fromIntegral (nobs ols - k_constant) / fromIntegral df_resid)
--           dataRank = rank $ diag singularValues -- TODO
--           df_resid = nobs ols - dataRank

In [ ]:
df_resid

In [ ]:
rsquared ols True

In [ ]:
ess ols True

In [ ]:
:info predict

In [ ]:
disp 2 normalized_cov_params
print beta

In [ ]:
let k_constant = 1 -- based on the way we defined x0
    nobs = rows wexog
    df_model = dataRank - k_constant
    df_resid = nobs - dataRank

In [ ]:
print nobs
print dataRank
print df_model
print df_resid

In [ ]:
-- OLS constructor
--     fields for WLS:
--     what's going on with the kwarg check for offset?

-- def __init__(self, endog, exog=None, missing='none', hasconst=None,
--              **kwargs):
--     if "weights" in kwargs:
--         msg = ("Weights are not supported in OLS and will be ignored"
--                "An exception will be raised in the next version.")
--         warnings.warn(msg, ValueWarning)
--     super().__init__(endog, exog, missing=missing,
--                               hasconst=hasconst, **kwargs)
--     if "weights" in self._init_keys:
--         self._init_keys.remove("weights")

--     if type(self) is OLS:
--         self._check_kwargs(kwargs, ["offset"])

-- WLS constructor
--     fields for RegressionModel
--      
-- def __init__(self, endog, exog, weights=1., missing='none', hasconst=None,
--              **kwargs):
--     if type(self) is WLS:
--         self._check_kwargs(kwargs)
--     weights = np.array(weights)
--     if weights.shape == ():
--         if (missing == 'drop' and 'missing_idx' in kwargs and
--                 kwargs['missing_idx'] is not None):
--             # patsy may have truncated endog
--             weights = np.repeat(weights, len(kwargs['missing_idx']))
--         else:
--             weights = np.repeat(weights, len(endog))
--     # handle case that endog might be of len == 1
--     if len(weights) == 1:
--         weights = np.array([weights.squeeze()])
--     else:
--         weights = weights.squeeze()
--     super().__init__(endog, exog, missing=missing,
--                               weights=weights, hasconst=hasconst, **kwargs)
--     nobs = self.exog.shape[0]
--     weights = self.weights
--     if weights.size != nobs and weights.shape[0] != nobs:
--         raise ValueError('Weights must be scalar or same length as design')

-- RegressionModel constructor
--     fields for LikelihoodModel
--     pinv_wexog defaulted to None
--     fields defined in initialize():
--         wexog
--         wendog
--         nobs
--         _df_model -- set to None
--         _df_resid -- set to None
--         rank      -- set to None
--     
-- def __init__(self, endog, exog, **kwargs):
--     super().__init__(endog, exog, **kwargs)
--     self.pinv_wexog: Float64Array | None = None
--     self._data_attr.extend(['pinv_wexog', 'wendog', 'wexog', 'weights'])
-- def initialize(self):
--     """Initialize model components."""
--     self.wexog = self.whiten(self.exog)
--     self.wendog = self.whiten(self.endog)
--     # overwrite nobs from class Model:
--     self.nobs = float(self.wexog.shape[0])

--     self._df_model = None
--     self._df_resid = None
--     self.rank = None

-- LikelihoodModel constructor
--     fields from Model, initialize passes so need to determine the method resolution in derived classes

-- def __init__(self, endog, exog=None, **kwargs):
--     super().__init__(endog, exog, **kwargs)
--     self.initialize()

-- Model constructor
--     data
--     k_constant (from data)
--     exog
--     endog
--     _data_attr
--     _init_keys

-- def __init__(self, endog, exog=None, **kwargs):
--     missing = kwargs.pop('missing', 'none')
--     hasconst = kwargs.pop('hasconst', None)
--     self.data = self._handle_data(endog, exog, missing, hasconst,
--                                   **kwargs)
--     self.k_constant = self.data.k_constant
--     self.exog = self.data.exog
--     self.endog = self.data.endog
--     self._data_attr = []
--     self._data_attr.extend(['exog', 'endog', 'data.exog', 'data.endog'])
--     if 'formula' not in kwargs:  # will not be able to unpickle without these
--         self._data_attr.extend(['data.orig_endog', 'data.orig_exog'])
--     # store keys for extras if we need to recreate model instance
--     # we do not need 'missing', maybe we need 'hasconst'
--     self._init_keys = list(kwargs.keys())
--     if hasconst is not None:
--         self._init_keys.append('hasconst')



In [ ]:
        -- if method == "pinv":
        --     if not (hasattr(self, 'pinv_wexog') and
        --             hasattr(self, 'normalized_cov_params') and
        --             hasattr(self, 'rank')):

        --         self.pinv_wexog, singular_values = pinv_extended(self.wexog)
        --         self.normalized_cov_params = np.dot(
        --             self.pinv_wexog, np.transpose(self.pinv_wexog))

        --         # Cache these singular values for use later.
        --         self.wexog_singular_values = singular_values
        --         self.rank = np.linalg.matrix_rank(np.diag(singular_values))

        --     beta = np.dot(self.pinv_wexog, self.wendog)

        -- elif method == "qr":
        --     if not (hasattr(self, 'exog_Q') and
        --             hasattr(self, 'exog_R') and
        --             hasattr(self, 'normalized_cov_params') and
        --             hasattr(self, 'rank')):
        --         Q, R = np.linalg.qr(self.wexog)
        --         self.exog_Q, self.exog_R = Q, R
        --         self.normalized_cov_params = np.linalg.inv(np.dot(R.T, R))

        --         # Cache singular values from R.
        --         self.wexog_singular_values = np.linalg.svd(R, 0, 0)
        --         self.rank = np.linalg.matrix_rank(R)
        --     else:
        --         Q, R = self.exog_Q, self.exog_R
        --     # Needed for some covariance estimators, see GH #8157
        --     self.pinv_wexog = np.linalg.pinv(self.wexog)
        --     # used in ANOVA
        --     self.effects = effects = np.dot(Q.T, self.wendog)
        --     beta = np.linalg.solve(R, effects)
        -- else:
        --     raise ValueError('method has to be "pinv" or "qr"')

        -- if self._df_model is None:
        --     self._df_model = float(self.rank - self.k_constant)
        -- if self._df_resid is None:
        --     self.df_resid = self.nobs - self.rank

        -- if isinstance(self, OLS):
        --     lfit = OLSResults(
        --         self, beta,
        --         normalized_cov_params=self.normalized_cov_params,
        --         cov_type=cov_type, cov_kwds=cov_kwds, use_t=use_t)
        -- else:
        --     lfit = RegressionResults(
        --         self, beta,
        --         normalized_cov_params=self.normalized_cov_params,
        --         cov_type=cov_type, cov_kwds=cov_kwds, use_t=use_t,
        --         **kwargs)
        -- return RegressionResultsWrapper(lfit)

In [ ]:
-- Results Constructors

-- OLSResults -- no __init__

-- RegressionResults
--    see LikelihoodModelResults
--    _cache
--    _wexog_singular_values
--    df_model
--    df_resid
--    cov_type  set if cov_type=='nonrobust'
--    cov_kwds  set if cov_type=='nonrobust'
--    if cov_type!='nonrobust': adjusted use_t and call self.get_robustcov_results
--        this potentially modifies the following fields (note this would seem to overwrite a similar process in LikelihoodModelResults
--            cov_type
--            cov_kwds
--            use_t
--            cov_params_default
--            df_resid_inference
-- 
-- def __init__(self, model, params, normalized_cov_params=None, scale=1.,
--              cov_type='nonrobust', cov_kwds=None, use_t=None, **kwargs):
--     super().__init__(
--         model, params, normalized_cov_params, scale)

--     self._cache = {}
--     if hasattr(model, 'wexog_singular_values'):
--         self._wexog_singular_values = model.wexog_singular_values
--     else:
--         self._wexog_singular_values = None

--     self.df_model = model.df_model
--     self.df_resid = model.df_resid

--     if cov_type == 'nonrobust':
--         self.cov_type = 'nonrobust'
--         self.cov_kwds = {
--             'description': 'Standard Errors assume that the ' +
--             'covariance matrix of the errors is correctly ' +
--             'specified.'}
--         if use_t is None:
--             use_t = True  # TODO: class default
--         self.use_t = use_t
--     else:
--         if cov_kwds is None:
--             cov_kwds = {}
--         if 'use_t' in cov_kwds:
--             # TODO: we want to get rid of 'use_t' in cov_kwds
--             use_t_2 = cov_kwds.pop('use_t')
--             if use_t is None:
--                 use_t = use_t_2
--             # TODO: warn or not?
--         self.get_robustcov_results(cov_type=cov_type, use_self=True,
--                                    use_t=use_t, **cov_kwds)
--     for key in kwargs:
--         setattr(self, key, kwargs[key])

-- LikelihoodModelResults
--    normalized_cov_params
--    scale
--    _use_t set to False
--    use_t     only set if use_t is in kwargs
--    cov_type  set if cov_type is in kwargs and cov_type=='nonrobust'
--    cov_kwds  set if cov_type is in kwargs and cov_type=='nonrobust'
--    if 'cov_type' is in kwargs but cov_type!='nonrobust', then get_robustcov_results is called
--        this potentially modifies the following fields
--            cov_type
--            cov_kwds
--            use_t
--            cov_params_default
--            df_resid_inference

-- def __init__(self, model, params, normalized_cov_params=None, scale=1.,
--              **kwargs):
--     super().__init__(model, params)
--     self.normalized_cov_params = normalized_cov_params
--     self.scale = scale
--     self._use_t = False
--     # robust covariance
--     # We put cov_type in kwargs so subclasses can decide in fit whether to
--     # use this generic implementation
--     if 'use_t' in kwargs:
--         use_t = kwargs['use_t']
--         self.use_t = use_t if use_t is not None else False
--     if 'cov_type' in kwargs:
--         cov_type = kwargs.get('cov_type', 'nonrobust')
--         cov_kwds = kwargs.get('cov_kwds', {})

--         if cov_type == 'nonrobust':
--             self.cov_type = 'nonrobust'
--             self.cov_kwds = {'description': 'Standard Errors assume that the ' +
--                              'covariance matrix of the errors is correctly ' +
--                              'specified.'}
--         else:
--             from statsmodels.base.covtype import get_robustcov_results
--             if cov_kwds is None:
--                 cov_kwds = {}
--             use_t = self.use_t
--             # TODO: we should not need use_t in get_robustcov_results
--             get_robustcov_results(self, cov_type=cov_type, use_self=True,
--                                   use_t=use_t, **cov_kwds)

-- Results
--     __dict__ updated from kwargs
--     _data_attr__ set to empty list
--     _data_in_cache initialized with three values
--     params -- set in initialize using passed in value
--     model  -- set in initialize using passed in model
--     k_constant -- set in initialize using member of model
-- 
-- def __init__(self, model, params, **kwd):
--     self.__dict__.update(kwd)
--     self.initialize(model, params, **kwd)
--     self._data_attr = []
--     # Variables to clear from cache
--     self._data_in_cache = ['fittedvalues', 'resid', 'wresid']

-- def initialize(self, model, params, **kwargs):
--     """
--     Initialize (possibly re-initialize) a Results instance.

--     Parameters
--     ----------
--     model : Model
--         The model instance.
--     params : ndarray
--         The model parameters.
--     **kwargs
--         Any additional keyword arguments required to initialize the model.
--     """
--     self.params = params
--     self.model = model
--     if hasattr(model, 'k_constant'):
--         self.k_constant = model.k_constant
